In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score


# ---------------------------------------------------------
# 1. 파일 경로 설정 (사용자 환경에 맞게 수정하세요)
# ---------------------------------------------------------
ARFF_PATH = "/home/junhyung/study/Data_Analysis_with_CKKS/Cluster/DBSCAN_CKKS/desilo/dataset/Other_cluster/twodiamonds.arff"
CSV_PATH = "/home/junhyung/study/Data_Analysis_with_CKKS/Cluster/DBSCAN_CKKS/desilo/atom_fhe_heap_result_eps0.14_min6.csv"

# CSV 안에서 어떤 컬럼을 라벨로 쓸지 지정 (DBSCAN 결과 라벨 컬럼명)
LABEL_COLUMN = "FHE_Cluster"

# 노이즈로 표시된 라벨 값 (DBSCAN 관례상 -1)
NOISE_LABEL = -1

# 노이즈 포인트를 실루엣 계산에서 제외할지 여부
# (노이즈를 포함하면 값이 왜곡되는 경우가 많아 기본값은 True로 둡니다)
EXCLUDE_NOISE = True


def load_arff_to_pts(filepath):
    pts = []
    data_section = False
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('%'):
                continue
            if line.lower().startswith('@data'):
                data_section = True
                continue
            if data_section:
                line = line.replace('\t', ' ').replace(',', ' ')
                values = line.split()
                if len(values) < 2:
                    continue
                # 마지막 라벨을 제외한 X, Y, Z 좌표만 추출
                row = [float(v) for v in values[:-1]]
                pts.append(row)
    return np.array(pts, dtype=np.float64)


def compute_silhouette(points, labels, exclude_noise=True, noise_label=-1):
    """
    실루엣 계수를 계산.

    - exclude_noise=True 인 경우, 노이즈(-1)로 표시된 포인트는 계산에서 제외.
      (DBSCAN 결과에 노이즈를 포함하면 실루엣 값이 왜곡되므로
       일반적으로 제외하고 계산합니다.)
    - 군집이 2개 미만이면 실루엣 계수가 정의되지 않으므로 None 반환.

    반환값: (score, n_clusters_used, n_points_used)
    """
    labels = np.asarray(labels)

    if exclude_noise:
        mask = labels != noise_label
    else:
        mask = np.ones(len(labels), dtype=bool)

    used_points = points[mask]
    used_labels = labels[mask]

    n_unique = len(set(used_labels))

    if n_unique < 2 or len(used_points) < 2:
        return None, n_unique, len(used_points)

    score = silhouette_score(used_points, used_labels)
    return score, n_unique, len(used_points)


def main():
    print("데이터 및 결과 파일 로딩 중...")

    # 1. 데이터 로드
    pts = load_arff_to_pts(ARFF_PATH)
    df = pd.read_csv(CSV_PATH)

    if len(pts) != len(df):
        print("🚨 경고: ARFF 데이터 개수와 CSV 결과 개수가 다릅니다!")
        return

    if LABEL_COLUMN not in df.columns:
        print(f"🚨 경고: CSV에 '{LABEL_COLUMN}' 컬럼이 없습니다. 사용 가능한 컬럼: {list(df.columns)}")
        return

    labels = df[LABEL_COLUMN].values

    # 2. 실루엣 계수 계산
    score, n_clusters, n_used = compute_silhouette(
        pts, labels,
        exclude_noise=EXCLUDE_NOISE,
        noise_label=NOISE_LABEL
    )

    print("\n===== Silhouette Coefficient =====")
    print(f"CSV 파일        : {CSV_PATH}")
    print(f"라벨 컬럼       : {LABEL_COLUMN}")
    print(f"노이즈 제외     : {EXCLUDE_NOISE}")
    print(f"사용된 군집 수  : {n_clusters}")
    print(f"사용된 포인트 수: {n_used} / {len(pts)}")

    if score is None:
        print("Silhouette score: N/A (군집이 2개 미만이라 정의되지 않음)")
    else:
        print(f"Silhouette score: {score:.4f}")
    print("===================================\n")


if __name__ == '__main__':
    main()

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


데이터 및 결과 파일 로딩 중...
🚨 경고: ARFF 데이터 개수와 CSV 결과 개수가 다릅니다!
